<style>
ul { padding-left:24px; }
table { margin-left:24px !important; margin-right:0 !important; }
th, td { text-align:left !important; }
li table { margin-left:0 !important; }
</style>

# 입자 군집 최적화 (Particle Swarm Optimization, PSO)

## 1) 개념

- 여러 입자가 개인 최적 위치와 군집(swarm)의 전역 최적 위치를 이용하여 해를 탐색하는 메타휴리스틱(metaheuristic)
- 목적함수(objective function)로 후보해(candidate solution) 평가, 도함수 계산 불필요
- 최소화 문제에서는 탐색한 위치 중 목적함수값이 가장 작은 위치를 결과로 선택

## 2) 용어

- 입자(particle): 후보해 하나, 각 의사결정변수를 성분으로 하는 위치 벡터로 표현
- 속도 벡터(velocity vector): 입자의 이동 방향과 크기를 결정하는 벡터
- 개인 최적 위치(personal best position, pbest): 해당 입자가 현재까지 탐색한 위치 중 목적함수값이 가장 작은 위치
- 전역 최적 위치(global best position, gbest): 전체 입자가 현재까지 탐색한 위치 중 목적함수값이 가장 작은 위치, 여기서 전역은 군집 전체의 탐색 기록을 의미

- 관성항(inertia term): 이전 속도 벡터에 관성 가중치를 곱한 항
- 인지항(cognitive term): 개인 최적 위치에서 현재 위치를 뺀 벡터에 인지 계수와 난수를 곱한 항
- 사회항(social term): 전역 최적 위치에서 현재 위치를 뺀 벡터에 사회 계수와 난수를 곱한 항

- 보라·파랑·초록 화살표는 관성항·인지항·사회항, 진한 화살표는 갱신된 속도 벡터, ‘Next’는 다음 위치, 점선은 벡터 합의 보조선

<div style="margin-left:24px;">
<img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/06-1_metaheuristic_optimization_particle_swarm_optimization/assets/movement.svg" alt="현재 위치에서 관성항·인지항·사회항을 더한 진한 화살표가 다음 위치를 가리킨다. 점선은 벡터 합의 보조선이다." width="440" style="display: block; width: 27.5em; max-width: 100%; height: auto; margin: 0;">
</div>

<style>
ul { padding-left:24px; }
table { margin-left:24px !important; margin-right:0 !important; }
th, td { text-align:left !important; }
li table { margin-left:0 !important; }
</style>

## 3) 방법론

<ol style="list-style:none;padding-left:24px;">
<li style="margin-bottom:10px;padding-left:2em;text-indent:-2em;">(1) 탐색 범위 안에 입자를 초기화하고 초기 속도 벡터를 난수로 설정, 목적함수 평가 후 pbest와 gbest 초기화</li>
<li style="margin-bottom:10px;padding-left:2em;text-indent:-2em;">(2) 동일한 gbest를 이용해 관성항·인지항·사회항을 합산하여 각 입자의 속도 벡터 갱신</li>
<li style="margin-bottom:10px;padding-left:2em;text-indent:-2em;">(3) 위치 벡터에 갱신된 속도 벡터를 더하여 각 입자의 위치 갱신</li>
<li style="margin-bottom:10px;padding-left:2em;text-indent:-2em;">(4) 경계 처리(boundary handling): 탐색 범위를 벗어난 성분을 경계값으로 제한하고 해당 속도 성분을 0으로 설정, 수정한 위치의 목적함수값 계산</li>
<li style="margin-bottom:10px;padding-left:2em;text-indent:-2em;">(5) 새 위치의 목적함수값이 더 작을 때 pbest 갱신, 모든 pbest 중 목적함수값이 가장 작은 위치를 gbest로 선택</li>
<li style="margin-bottom:10px;padding-left:2em;text-indent:-2em;">(6) 종료 조건(stopping criterion)인 최대 반복 횟수까지 (2)~(5) 반복, 마지막 목적함수 평가와 최적 위치 갱신 후 gbest 반환</li>
</ol>

<div style="margin-left:24px;">
<img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/06-1_metaheuristic_optimization_particle_swarm_optimization/assets/pso_method_flow.svg" alt="입자 초기화, 속도와 위치 갱신, 경계 처리, 목적함수 평가, 개인 최적 위치와 전역 최적 위치 갱신 후 최대 반복 횟수에서 종료" width="440" style="display:block;width:27.5em;max-width:100%;height:auto;">
</div>

- 관성 가중치(inertia weight)는 이전 속도의 반영 정도, 인지 계수(cognitive coefficient)와 사회 계수(social coefficient)는 각각 인지항과 사회항의 크기 조절
- 인지항과 사회항에는 입자·반복·성분별로 독립 추출한 0 이상 1 미만의 균등분포(uniform distribution) 난수를 사용

## 4) 구현

- 순서도의 초기화 → 속도·위치 갱신 → 경계 처리 → 평가 → pbest·gbest 갱신을 공통 함수로 구현, 최종 gbest 반환
- 이 구현에서는 초기 속도의 각 성분을 해당 탐색 구간 폭의 −10%~10%에서 균등분포 난수로 설정


In [ ]:
import numpy as np


def particle_swarm(initial_positions, objective, lower, upper, iterations=20, seed=0):
    # 난수 생성기와 초기 위치 설정
    rng = np.random.default_rng(seed)
    positions = np.array(initial_positions, dtype=float, copy=True)
    if np.any(positions < lower) or np.any(positions > upper):
        raise ValueError("Initial positions must satisfy the bounds.")

    # 탐색 구간 폭에 비례하는 초기 속도 설정
    span = np.asarray(upper) - np.asarray(lower)
    velocities = rng.uniform(-0.1, 0.1, positions.shape) * span

    # 초기 위치를 평가하여 개인 및 전역 최적 위치 설정
    pbest = positions.copy()
    best_values = np.array([objective(x) for x in positions])
    gbest = pbest[np.argmin(best_values)].copy()

    for _ in range(iterations):
        # 관성항, 인지항, 사회항으로 속도 갱신
        r1 = rng.random(positions.shape)
        r2 = rng.random(positions.shape)
        inertia = 0.6 * velocities
        cognitive_term = 1.4 * r1 * (pbest - positions)
        social_term = 1.4 * r2 * (gbest - positions)
        velocities = inertia + cognitive_term + social_term

        # 위치 이동 후 경계값으로 제한, 벗어난 성분의 속도는 0으로 설정
        proposed = positions + velocities
        positions = np.clip(proposed, lower, upper)
        velocities[positions != proposed] = 0.0

        # 이동한 위치의 목적함수값 계산
        values = np.array([objective(x) for x in positions])

        # 기존 기록보다 좋은 위치로 개인 최적 위치 갱신
        improved = values < best_values
        pbest[improved] = positions[improved]
        best_values[improved] = values[improved]

        # 모든 개인 최적 위치 중 가장 좋은 위치 선택
        gbest = pbest[np.argmin(best_values)].copy()

    # 반복 종료 후 전역 최적 위치 반환
    return gbest


<style>
ul { padding-left:24px; }
table { margin-left:24px !important; margin-right:0 !important; }
th, td { text-align:left !important; }
li table { margin-left:0 !important; }
</style>

## 5) 예제

- 이진 예제의 의사결정벡터 $\mathbf{x}=(x_1,x_2,x_3,x_4)$는 네 기능의 선택 여부, 1은 켜기·0은 끄기
- 입자는 실수 위치 $\mathbf{z}=(z_1,z_2,z_3,z_4)$로 0~1에서 이동하고 평가 시 0.5 미만은 0·이상은 1로 변환, 켠 기능 수의 음수를 최소화하며 20회 반복


In [ ]:
# 이진 예제: 실수 위치를 기능 선택 여부로 변환
def decode(position):
    return (np.asarray(position) >= 0.5).astype(int)

# 켠 기능 수 최대화를 음수의 최소화로 변환
def binary_objective(position):
    return -int(decode(position).sum())

# 초기 후보 설정 및 PSO 실행
binary_positions = [[1, 0, 1, 0], [0, 1, 0, 1],
                    [1, 0, 0, 0], [0, 0, 1, 0]]
binary_result = particle_swarm(binary_positions, binary_objective, 0, 1, iterations=20)

# 탐색 결과를 선택 여부로 변환하고 출력
binary_decision = decode(binary_result)
print("Binary decision:", binary_decision)
print("Active functions:", int(binary_decision.sum()))


<style>
ul { padding-left:24px; }
table { margin-left:24px !important; margin-right:0 !important; }
th, td { text-align:left !important; }
li table { margin-left:0 !important; }
</style>

- 실수 예제는 두 좌표를 `(0.5, -0.5)`에 가깝게 조정, 각 좌표 범위 −2~2, 거리 제곱을 최소화하며 40회 반복


In [ ]:
# 실수 예제: 목표 좌표까지의 거리 제곱을 최소화
def objective(decision):
    return (decision[0] - 0.5)**2 + (decision[1] + 0.5)**2

# 초기 후보 설정 및 PSO 실행
initial_positions = [[-1.0, -1.0], [-1.0, 1.0],
                     [1.0, -1.0], [1.0, 1.0]]
result = particle_swarm(initial_positions, objective, [-2, -2], [2, 2], iterations=40)

# 최종 위치와 목적함수값 출력
print("Best position:", result)
print("Objective value:", objective(result))
print("Stop reason: iteration limit")
